In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph
import numpy as np
from scipy.cluster.hierarchy import linkage
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
import numpy as np
from src.clustering import get_average_random_pair_similarity, check_coarse_elbow, plot_elbow_curve, plot_hierarchy_tree, build_dynamic_hierarchy, get_cluster_metrics_table, save_artifacts, apply_hierarchy_to_queries, format_hierarchical_metrics

In [4]:
# 1. Load the embeddings
doc_emb = np.load("../data/artifacts/doc_emb.npy")
query_emb = np.load("../data/artifacts/query_emb.npy")

# 2. Calculate baseline similarity
avg_sim_bge = get_average_random_pair_similarity(doc_emb, num_pairs=10000)
print(f"BGE EMBEDDING: Average cosine similarity for 10,000 random pairs: {avg_sim_bge:.4f}")

BGE EMBEDDING: Average cosine similarity for 10,000 random pairs: 0.3736


In [ ]:
# Check elbow on a sample to decide our Coarse K
k_options = [20, 30, 40, 50, 60, 70, 80]
elbow_df = check_coarse_elbow(doc_emb, k_options, sample_size=50000)

# Plot it
fig = plot_elbow_curve(elbow_df)
fig.show()

In [12]:
CHOSEN_COARSE_K = 40  
fine_options = [2, 3, 4, 5, 6, 7, 8, 9, 10]

# Run the full hierarchy
results = build_dynamic_hierarchy(
    doc_emb=doc_emb, 
    chosen_coarse_k=CHOSEN_COARSE_K, 
    fine_k_options=fine_options
)

# View the new Dendrogram-style Icicle Tree
tree_fig = plot_hierarchy_tree(results["metrics_df"])
tree_fig.show()

L2 Normalizing complete document dataset...
Fitting full dataset to Coarse K=40...

Optimizing Fine K for each of the 40 coarse clusters...

Full Hierarchy Build Complete!


In [14]:
# Predict query mappings
q_coarse, q_hierarchical = apply_hierarchy_to_queries(
    query_emb=query_emb,
    coarse_model=results["coarse_model"],
    fine_models=results["fine_models"]
)

# Save everything to disk
save_artifacts(
    save_dir="../data/artifacts/clustering_output",
    doc_labels=results["doc_hierarchical_labels"],
    query_labels=q_hierarchical,
    coarse_model=results["coarse_model"],
    fine_models=results["fine_models"]
)

All artifacts successfully saved to ../data/artifacts/clustering_output/


In [23]:
from sklearn.preprocessing import normalize

# 1. L2-Normalize your notebook's embeddings (Metrics table expects cosine space!)
doc_emb_normalized = normalize(doc_emb, norm='l2')

# 2. Extract the coarse labels from the results dictionary you generated earlier
my_coarse_labels = results["doc_coarse_labels"]

# Extract the raw metrics table from your pipeline results
raw_df = results["metrics_df"]

# Apply the formatting and display it
formatted_table = format_hierarchical_metrics(raw_df)

# The display() function will render the styled HTML table in Jupyter
display(formatted_table)

In [29]:
import plotly.express as px

# ==========================================
# Chart 1: Fine Clusters per Coarse Cluster
# ==========================================
# Extract the Coarse Parent (e.g., "C0" from "C0_F1")
df_fine["Coarse Parent"] = df_fine["Cluster ID"].apply(lambda x: x.split("_")[0])

# Count how many fine clusters belong to each coarse parent
fine_cluster_counts = df_fine.groupby("Coarse Parent").size().reset_index(name="Fine_Count")

# Plot the distribution
fig1 = px.histogram(
    fine_cluster_counts, 
    x="Fine_Count", 
    title="Distribution: How many Fine Clusters per Coarse Cluster?",
    labels={"Fine_Count": "Number of Fine Clusters"},
    text_auto=True,
    color_discrete_sequence=['#D6BEFA']
)

# Update layout to treat the counts as discrete categories
fig1.update_layout(
    yaxis_title="Number of Coarse Clusters", 
    xaxis=dict(
        type='category',                     # This creates separate, distinct buckets
        categoryorder='category ascending'   # Ensures they display from lowest to highest
    )
)
fig1.show()

In [28]:
# ==========================================
# Chart 2: Documents per Fine Cluster
# ==========================================
# Plot the distribution of cluster sizes (How many documents end up in a fine cluster?)
fig2 = px.histogram(
    df_fine, 
    x="Size", 
    title="Distribution: Number of Documents per Fine Cluster",
    labels={"Size": "Documents per Fine Cluster"},
    nbins=30, # Adjust this if you want wider/narrower bins
    color_discrete_sequence=['#CB94F7']
)
fig2.update_layout(yaxis_title="Number of Fine Clusters")
fig2.show()

In [32]:
query_df = pd.read_parquet("../data/interim/questions.parquet")
docs_df = pd.read_parquet("../data/interim/nodes.parquet")

In [33]:
import pandas as pd
import numpy as np
import ast
import plotly.express as px

# 1. Map Node IDs to their Cluster Labels
# We assume results["doc_coarse_labels"] matches the index/order of docs_df
node_to_coarse = dict(zip(docs_df['node_id'], results['doc_coarse_labels']))
node_to_fine = dict(zip(docs_df['node_id'], results['doc_hierarchical_labels']))

# Helper function to safely parse lists and count unique clusters
def count_spanned_clusters(row):
    gold_nodes = row['gold_node_ids']
    
    # Safely evaluate if pandas loaded the list as a string
    if isinstance(gold_nodes, str):
        try:
            gold_nodes = ast.literal_eval(gold_nodes)
        except:
            gold_nodes = [] # Fallback
            
    # Collect all unique clusters these gold nodes belong to
    coarse_set = set()
    fine_set = set()
    
    for node in gold_nodes:
        if node in node_to_coarse:
            coarse_set.add(node_to_coarse[node])
            fine_set.add(node_to_fine[node])
            
    return pd.Series({
        'spanned_coarse_count': len(coarse_set),
        'spanned_fine_count': len(fine_set)
    })

# 2. Apply the counting logic to your query dataframe
print("Calculating cluster spread for queries...")
query_df[['spanned_coarse_count', 'spanned_fine_count']] = query_df.apply(count_spanned_clusters, axis=1)

# Filter out any rows that didn't match or where n_hops isn't what we want
df_plot = query_df[query_df['n_hops'].isin([2, 3, 4])].copy()

Calculating cluster spread for queries...


In [34]:
# ==========================================
# Chart 1: Coarse Cluster Spread per Query
# ==========================================
fig1 = px.histogram(
    df_plot,
    x='spanned_coarse_count',
    facet_col='n_hops',          # Creates separate side-by-side plots for 2, 3, 4
    color='n_hops',              # Colors them differently for easy reading
    title="Coarse Fragmentation: How many distinct Coarse Clusters do Gold Nodes span?",
    labels={'spanned_coarse_count': 'Distinct Coarse Clusters', 'n_hops': 'Hops'},
    text_auto=True
)

# Force X-axis to be discrete categorical buckets (1, 2, 3...)
fig1.update_xaxes(type='category', categoryorder='category ascending')
fig1.update_layout(showlegend=False, yaxis_title="Number of Queries")
fig1.show()


# ==========================================
# Chart 2: Fine Cluster Spread (ONLY for Queries contained in 1 Coarse Cluster)
# ==========================================
# Filter strictly for queries where all gold nodes fell into a SINGLE coarse cluster
df_contained = df_plot[df_plot['spanned_coarse_count'] == 1].copy()

fig2 = px.histogram(
    df_contained,
    x='spanned_fine_count',
    facet_col='n_hops',
    color='n_hops',
    title="Fine Fragmentation: Spanned Fine Clusters (For Gold Nodes contained in 1 Coarse Cluster)",
    labels={'spanned_fine_count': 'Distinct Fine Clusters', 'n_hops': 'Hops'},
    text_auto=True
)

fig2.update_xaxes(type='category', categoryorder='category ascending')
fig2.update_layout(showlegend=False, yaxis_title="Number of Queries")
fig2.show()

In [35]:
import ast
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import normalize

# ==========================================
# 1. Extract Centroids & Compute All Ranks
# ==========================================
print("Extracting centroids and calculating query-to-cluster ranks...")

# Coarse Centroids
coarse_centroids = results["coarse_model"].cluster_centers_

# Fine Centroids (Flattened globally for absolute ranking)
fine_ids = []
fine_centroids_list = []
for c_id, f_model in results["fine_models"].items():
    for f_label, centroid in enumerate(f_model.cluster_centers_):
        fine_ids.append(f"C{c_id}_F{f_label}")
        fine_centroids_list.append(centroid)
        
fine_centroids = np.array(fine_centroids_list)

# Normalize queries for Cosine Similarity
queries_norm = normalize(query_emb, norm='l2')

# Matrix multiplication for instant Cosine Similarities
coarse_sims = queries_norm @ coarse_centroids.T
fine_sims = queries_norm @ fine_centroids.T

# Magic numpy trick to convert similarities to Ranks (1 = highest similarity)
coarse_ranks_matrix = np.argsort(np.argsort(-coarse_sims, axis=1), axis=1) + 1
fine_ranks_matrix = np.argsort(np.argsort(-fine_sims, axis=1), axis=1) + 1


# ==========================================
# 2. Map Gold Nodes to their Ranks
# ==========================================
print("Mapping gold nodes to their cluster ranks...")

# Dictionaries to quickly look up which cluster a node belongs to
node_to_coarse = dict(zip(docs_df['node_id'], results['doc_coarse_labels']))
node_to_fine = dict(zip(docs_df['node_id'], results['doc_hierarchical_labels']))

rank_records = []

for idx, row in query_df.iterrows():
    n_hops = row['n_hops']
    if n_hops not in [2, 3, 4]: 
        continue
        
    gold_nodes = row['gold_node_ids']
    if isinstance(gold_nodes, str):
        try:
            gold_nodes = ast.literal_eval(gold_nodes)
        except:
            gold_nodes = []
            
    # Find unique clusters that contain the gold nodes for this specific query
    coarse_set = set()
    fine_set = set()
    for node in gold_nodes:
        if node in node_to_coarse:
            coarse_set.add(node_to_coarse[node])
            fine_set.add(node_to_fine[node])
            
    # Record the rank of EVERY unique coarse cluster required to answer the query
    for c_id in coarse_set:
        c_rank = coarse_ranks_matrix[idx, c_id]
        rank_records.append({'n_hops': n_hops, 'Level': 'Coarse', 'Rank': c_rank})
        
    # Record the rank of EVERY unique fine cluster required
    for f_id in fine_set:
        f_matrix_idx = fine_ids.index(f_id)
        f_rank = fine_ranks_matrix[idx, f_matrix_idx]
        rank_records.append({'n_hops': n_hops, 'Level': 'Fine', 'Rank': f_rank})

df_ranks = pd.DataFrame(rank_records)


# ==========================================
# 3. Plot Histograms
# ==========================================
print("Generating Plots...")

# Chart 1: Coarse Cluster Ranks
df_coarse_ranks = df_ranks[df_ranks['Level'] == 'Coarse']
fig1 = px.histogram(
    df_coarse_ranks,
    x='Rank',
    facet_col='n_hops',
    color='n_hops',
    title="Coarse Cluster Retrieval: Rank of Gold Clusters (1 = Best)",
    labels={'Rank': 'Rank of Gold Coarse Cluster', 'n_hops': 'Hops'},
    nbins=40 # Assuming you have roughly 40 coarse clusters
)
fig1.update_layout(showlegend=False, yaxis_title="Count of Gold Clusters")
fig1.show()


# Chart 2: Fine Cluster Ranks
df_fine_ranks = df_ranks[df_ranks['Level'] == 'Fine']
fig2 = px.histogram(
    df_fine_ranks,
    x='Rank',
    facet_col='n_hops',
    color='n_hops',
    title="Fine Cluster Retrieval: Global Rank of Gold Fine Clusters (Out of all ~400)",
    labels={'Rank': 'Global Rank of Gold Fine Cluster', 'n_hops': 'Hops'},
    nbins=50
)
fig2.update_layout(showlegend=False, yaxis_title="Count of Gold Clusters")
fig2.show()

Extracting centroids and calculating query-to-cluster ranks...
Mapping gold nodes to their cluster ranks...
Generating Plots...
